In [ ]:

import pandas as pd
import numpy as np


# ==========================================
# 1. Load and clean dataset
# ==========================================

df_clean = pd.read_csv("digikala_comments_clean_ids.csv")

df_base = df_clean.drop_duplicates(
    subset="id",
    keep="first"
).copy()

df_base = df_base.dropna(
    subset=["id", "product_id", "recommendation_status_clean", "text"]
).copy()

df_base["recommendation_status_clean"] = (
    df_base["recommendation_status_clean"].astype(str).str.strip()
)

df_base["text"] = df_base["text"].astype(str).str.strip()

df_base = df_base[
    (df_base["recommendation_status_clean"] != "") &
    (df_base["text"] != "")
].copy()

df_base = df_base.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)


# ==========================================
# 2. Classes and targets
# ==========================================

classes = sorted(df_base["recommendation_status_clean"].unique())

if len(classes) != 3:
    raise ValueError(f"Expected 3 classes, found: {classes}")

target_balanced = pd.Series(
    [33334, 33333, 33333],
    index=classes
)

class_ratios = (
    df_base["recommendation_status_clean"]
    .value_counts(normalize=True)
    .reindex(classes)
)

target_natural = (
    class_ratios * 100000
).round().astype(int)

target_natural.iloc[0] += 100000 - target_natural.sum()


# ==========================================
# 3. Group products and count classes
# ==========================================

product_groups = {
    pid: group
    for pid, group in df_base.groupby("product_id")
}

product_counts = pd.DataFrame({
    pid: group["recommendation_status_clean"].value_counts()
    for pid, group in product_groups.items()
}).T.reindex(columns=classes, fill_value=0)

product_counts = product_counts.fillna(0).astype(int)

product_ids = list(product_groups.keys())


# ==========================================
# 4. Select product groups
# ==========================================

def select_products(
    available_ids,
    targets,
    seed,
    max_attempts=1000
):
    rng = np.random.default_rng(seed)

    available_ids = list(available_ids)

    for attempt in range(max_attempts):

        shuffled = available_ids.copy()
        rng.shuffle(shuffled)

        selected = []
        counts = pd.Series(0, index=classes, dtype=int)

        for pid in shuffled:

            if (counts >= targets).all():
                break

            group_counts = product_counts.loc[pid]

            remaining = (targets - counts).clip(lower=0)

            gain = np.minimum(
                group_counts,
                remaining
            ).sum()

            if gain > 0:
                selected.append(pid)

                counts += np.minimum(
                    group_counts,
                    remaining
                )

        if (counts >= targets).all():
            return selected

    raise ValueError(
        "Could not find enough product groups for these targets. "
        "The requested split may not be feasible."
    )


# ==========================================
# 5. Build balanced train and test
# ==========================================

selected_train_balanced = select_products(
    product_ids,
    target_balanced,
    seed=42
)

used_1 = set(selected_train_balanced)

selected_test_balanced = select_products(
    [pid for pid in product_ids if pid not in used_1],
    target_balanced,
    seed=43
)

used_2 = used_1 | set(selected_test_balanced)


# ==========================================
# 6. Build natural train and test
# ==========================================

selected_train_natural = select_products(
    [pid for pid in product_ids if pid not in used_2],
    target_natural,
    seed=44
)

used_3 = used_2 | set(selected_train_natural)

selected_test_natural = select_products(
    [pid for pid in product_ids if pid not in used_3],
    target_natural,
    seed=45
)


# ==========================================
# 7. Create DataFrames
# ==========================================

def make_dataset(selected_ids, targets, seed):

    candidates = df_base[
        df_base["product_id"].isin(selected_ids)
    ]

    result = pd.concat(
        [
            candidates[
                candidates["recommendation_status_clean"] == label
            ].sample(
                n=int(targets[label]),
                random_state=seed
            )
            for label in classes
        ],
        ignore_index=True
    )

    return result.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)


df_train_balanced = make_dataset(
    selected_train_balanced,
    target_balanced,
    42
)

df_test_balanced = make_dataset(
    selected_test_balanced,
    target_balanced,
    43
)

df_train_natural = make_dataset(
    selected_train_natural,
    target_natural,
    44
)

df_test_natural = make_dataset(
    selected_test_natural,
    target_natural,
    45
)


# ==========================================
# 8. Check sizes and distributions
# ==========================================

datasets = {
    "Train Balanced": df_train_balanced,
    "Test Balanced": df_test_balanced,
    "Train Natural": df_train_natural,
    "Test Natural": df_test_natural
}

for name, df in datasets.items():

    print("\n==========", name, "==========")

    print("Total:", len(df))

    print(df["recommendation_status_clean"].value_counts())

    print("\nPercentage:")
    print(
        df["recommendation_status_clean"]
        .value_counts(normalize=True)
        .mul(100)
    )


# ==========================================
# 9. Check product overlap
# ==========================================

dataset_names = list(datasets.keys())

for i in range(len(dataset_names)):
    for j in range(i + 1, len(dataset_names)):

        name_1 = dataset_names[i]
        name_2 = dataset_names[j]

        ids_1 = set(datasets[name_1]["product_id"])
        ids_2 = set(datasets[name_2]["product_id"])

        overlap = ids_1 & ids_2

        print(
            f"\n{name_1} - {name_2} overlap:",
            len(overlap)
        )

        assert len(overlap) == 0


# Check exact sizes
for name, df in datasets.items():
    assert len(df) == 100000, f"{name} has wrong size!"


# ==========================================
# 10. Save datasets
# ==========================================

df_train_balanced.to_csv(
    "train_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)

df_test_balanced.to_csv(
    "test_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)

df_train_natural.to_csv(
    "train_natural.csv",
    index=False,
    encoding="utf-8-sig"
)

df_test_natural.to_csv(
    "test_natural.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAll four datasets saved successfully!")



========== Train Balanced ==========
Total: 100000
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Percentage:
recommendation_status_clean
no_idea            33.334
recommended        33.333
not_recommended    33.333
Name: proportion, dtype: float64

========== Test Balanced ==========
Total: 100000
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Percentage:
recommendation_status_clean
no_idea            33.334
recommended        33.333
not_recommended    33.333
Name: proportion, dtype: float64

========== Train Natural ==========
Total: 100000
recommendation_status_clean
recommended        76025
no_idea            12252
not_recommended    11723
Name: count, dtype: int64

Percentage:
recommendation_status_clean
recommended        76.025
no_idea            12.252
not_recommended    11.723
Name: proportion, dtype: float64

==

In [ ]:

# ==========================================
# 1. Load existing test datasets
# ==========================================

df_test_blanced = pd.read_csv("test_balanced.csv")
df_test_natural = pd.read_csv("test_natural.csv")


# ==========================================
# 2. Load original dataset
# ==========================================

df_clean = pd.read_csv("digikala_comments_clean_ids.csv")

df_base = df_clean.drop_duplicates(
    subset="id",
    keep="first"
).copy()

df_base = df_base.dropna(
    subset=[
        "id",
        "product_id",
        "recommendation_status_clean",
        "text"
    ]
).copy()

df_base["recommendation_status_clean"] = (
    df_base["recommendation_status_clean"]
    .astype(str)
    .str.strip()
)

df_base["text"] = (
    df_base["text"]
    .astype(str)
    .str.strip()
)

df_base = df_base[
    df_base["recommendation_status_clean"].ne("") &
    df_base["text"].ne("")
].copy()

df_base = df_base.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)


# ==========================================
# 3. Extract product IDs from test datasets
# ==========================================

balanced_test_products = set(
    df_test_blanced["product_id"].dropna().unique()
)

natural_test_products = set(
    df_test_natural["product_id"].dropna().unique()
)

print("Balanced test products:", len(balanced_test_products))
print("Natural test products:", len(natural_test_products))


# ==========================================
# 4. Create training datasets
# ==========================================

df_train_balanced = df_base[
    ~df_base["product_id"].isin(balanced_test_products)
].copy()

df_train_natural = df_base[
    ~df_base["product_id"].isin(natural_test_products)
].copy()


# ==========================================
# 5. Check product overlap
# ==========================================

balanced_overlap = (
    set(df_train_balanced["product_id"]) &
    balanced_test_products
)

natural_overlap = (
    set(df_train_natural["product_id"]) &
    natural_test_products
)

assert len(balanced_overlap) == 0
assert len(natural_overlap) == 0

print("\nBalanced product overlap:", len(balanced_overlap))
print("Natural product overlap:", len(natural_overlap))


# ==========================================
# 6. Check dataset sizes
# ==========================================

datasets = {
    "Train Balanced": df_train_balanced,
    "Test Balanced": df_test_blanced,
    "Train Natural": df_train_natural,
    "Test Natural": df_test_natural
}

for name, data in datasets.items():

    print("\n==========", name, "==========")

    print("Total records:", len(data))
    print("Unique texts:", data["text"].nunique())
    print("Unique products:", data["product_id"].nunique())

    print("\nClass distribution:")
    print(data["recommendation_status_clean"].value_counts())

    print("\nClass percentages:")
    print(
        data["recommendation_status_clean"]
        .value_counts(normalize=True)
        .mul(100).round(2)
    )


# ==========================================
# 7. Save training datasets
# ==========================================

df_train_balanced.to_csv(
    "train_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)

df_train_natural.to_csv(
    "train_natural.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nBoth training datasets saved successfully!")


Balanced test products: 13164
Natural test products: 7844

Balanced product overlap: 0
Natural product overlap: 0

========== Train Balanced ==========
Total records: 3639394
Unique texts: 3639394
Unique products: 271834

Class distribution:
recommendation_status_clean
recommended        2771263
no_idea             443875
not_recommended     424256
Name: count, dtype: int64

Class percentages:
recommendation_status_clean
recommended        76.15
no_idea            12.20
not_recommended    11.66
Name: proportion, dtype: float64

========== Test Balanced ==========
Total records: 100000
Unique texts: 100000
Unique products: 13164

Class distribution:
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Class percentages:
recommendation_status_clean
no_idea            33.33
recommended        33.33
not_recommended    33.33
Name: proportion, dtype: float64

========== Train Natural ==========
Total records: 380049